# Task 3：ResNet34 五折对照——320、翻转 TTA、512

本轮固定 ResNet34，比较三种方案：**320 原图、320 水平翻转 TTA、512 原图**。两个分辨率各训练五折，每折 80 轮；TTA 复用 320 的同一批权重，因此实际训练 10 个模型。

80 张训练图按相同的完全重复图片组、分辨率分层分成五折，每折约 64 张训练/16 张验证。`0039` 与 `0077` 必须同折。没有患者 ID，尚不能保证患者层面的独立性。

**当前公开成绩参考：**此前 ResNet34 单模型全量版，公开榜得分为 12.31624。本轮尚无实验或比赛成绩，不能把预计改进写成已验证结论。

**运行位置：Kaggle GPU。** 本地有 CPU 实验且可用内存不足，此 Notebook 默认阻止本地执行。Notebook 内含全部函数，不需要上传 Python 脚本。依赖 `torch torchvision numpy pandas pillow`，Kaggle 通常已提供；不需要 timm。若没挂载 ImageNet 权重，首次运行需开启 Internet 下载约 87 MB 的 [torchvision 官方 ResNet34 权重](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet34.html)。

官方 Evaluation 公式仍未确认，默认按坐标 RMSE 保存和比较主模型，同时保留平均欧氏距离最优权重和指标。需要改指标时，应在首次运行前修改配置并使用新输出目录。


## 1. 配置与资源保护

在 Kaggle 添加比赛数据，启用 GPU，首次下载权重时启用 Internet；建议 **Save Version → Save & Run All**。即使可见两张 GPU，本实现也只使用 `cuda:0`，各折顺序运行。

`RUN_SIZES`、`RUN_FOLDS` 只控制本次执行哪些任务，不改变完整实验协议。可先设 `RUN_SIZES=[320]` 得到第一批完整五折结果，之后改回 `[320,512]`，已完成折会直接复用。配置、数据或生成版本冲突会停止。自行修改模型/训练实现时请使用新的输出目录，不要依赖原 Notebook 中固定的生成版本指纹。


In [ ]:
from pathlib import Path

if not Path('/kaggle/working').is_dir():
    raise RuntimeError('请在 Kaggle GPU 环境运行。本机 CPU 实验仍在使用内存，本 Notebook 不在本地启动。')

DATA_DIR = None                   # 自动发现 detection；数据有多份时填写完整路径
OUTPUT_DIR = Path('/kaggle/working/task3_resnet34_cv')
RESUME_DIR = None                 # 可填挂载的此前 Notebook 输出目录，里面应直接包含 protocol.json
WEIGHTS = None                    # 可填挂载的 resnet34-b627a593.pth
EPOCHS = 80
BATCH_SIZE = 8                    # 两个分辨率保持相同
SEED = 42
SELECTION_METRIC = 'rmse_coordinate_px'  # 或 mean_distance_px；官方公式仍待确认
CHECKPOINT_EVERY = 5              # 中断后最多重跑约5轮（含被中断轮）
CPU_THREADS = 2
KEEP_LAST_CHECKPOINT = False      # 完整一折完成后仅删除其 optimizer last.pt；两份最佳权重保留
RUN_SIZES = [320, 512]
RUN_FOLDS = [0, 1, 2, 3, 4]


## 2. 数据与模型实现

与此前成功的 ResNet34 保持一致：ImageNet 初始化、相同热图解码器、固定编码器 BatchNorm 运行统计、相同图像增强和标准化。AdamW 编码器/定位头学习率 1e-4/1e-3，weight_decay=1e-4，余弦周期80轮、混合精度、梯度裁剪。各折种子为 `SEED+fold`，对应分辨率使用相同种子。

320 输出 80×80，标签高斯 σ=1.5，解码半径3；512 输出 128×128，σ=2.4，半径5（整数近似），尽量保持归一化监督宽度和解码邻域一致。这样分辨率实验不会同时无意改成更窄的监督标签。

TTA：水平翻转、逆转 X 坐标，再等权平均坐标；同一折使用原图指标选出的同一个 checkpoint，不另选 TTA 最佳轮次。


In [ ]:
from __future__ import annotations
from collections import defaultdict
from contextlib import contextmanager
import csv, gc, hashlib, json, math, os, random, shutil, statistics, sys, time, zipfile
import xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

def save_json(path, value):
    path = Path(path)
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(4)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def find_data(explicit=None):
    if explicit:
        candidates = [Path(explicit)]
    else:
        base = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
        candidates = [base / '2026task3/detection', Path.cwd() / '2026task3/detection',
                      Path.cwd() / 'task3/2026task3/detection']
        if Path('/kaggle/input').exists():
            candidates += [p.parent for p in Path('/kaggle/input').rglob('fovea_localization_train_GT.csv')]
    valid = sorted({p.resolve() for p in candidates
                    if (p / 'fovea_localization_train_GT.csv').is_file()
                    and (p / 'sample_submission.csv').is_file()
                    and (p / 'train').is_dir() and (p / 'test').is_dir()})
    if len(valid) != 1:
        raise ValueError(f'Expected one dataset; set data_dir explicitly. Found: {valid}')
    return valid[0]


def load_manifest(data):
    labels = pd.read_csv(data / 'fovea_localization_train_GT.csv')
    if list(labels.columns) != ['data', 'Fovea_X', 'Fovea_Y'] or labels.isna().any().any():
        raise ValueError('Invalid training CSV columns or missing values.')
    if labels['data'].duplicated().any():
        raise ValueError('Duplicate training IDs.')
    lookup = labels.set_index('data')
    records = []
    for subset in ['train', 'test']:
        files = sorted(p for p in (data / subset).iterdir()
                       if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
        for p in files:
            image_id = int(p.stem)
            with Image.open(p) as im:
                im = im.convert('RGB')
                im.load()
                w, h = im.size
                # Hash decoded pixels, not JPEG metadata. Size participates in the hash.
                digest = hashlib.sha256(f'{w},{h}:'.encode() + im.tobytes()).hexdigest()
            record = dict(image_id=image_id, subset=subset, filename=p.name,
                          width=w, height=h, pixel_sha256=digest)
            if subset == 'train':
                if image_id not in lookup.index:
                    raise ValueError(f'Missing label: {p.name}')
                x, y = lookup.loc[image_id, ['Fovea_X', 'Fovea_Y']].astype(float)
                if not np.isfinite([x, y]).all() or not (0 <= x < w and 0 <= y < h):
                    raise ValueError(f'Coordinate outside image: {p.name}')
                record.update(x=float(x), y=float(y), nx=float(x / w), ny=float(y / h))
                xml_path = data / 'train_location' / f'{image_id:04}.xml'
                if not xml_path.is_file():
                    raise ValueError(f'Missing XML: {xml_path}')
                xml = ET.parse(xml_path).getroot()
                if (int(xml.findtext('size/width')), int(xml.findtext('size/height'))) != (w, h):
                    raise ValueError(f'XML size mismatch: {p.name}')
                objects = [v for v in xml.findall('object') if v.findtext('name').lower() == 'fovea']
                if len(objects) != 1:
                    raise ValueError(f'Expected one Fovea box: {xml_path}')
                box = objects[0].find('bndbox')
                xmin, ymin, xmax, ymax = [float(box.findtext(k)) for k in ['xmin', 'ymin', 'xmax', 'ymax']]
                if not (xmin <= x <= xmax and ymin <= y <= ymax):
                    raise ValueError(f'CSV coordinate outside XML box: {p.name}')
                record.update(xml_center_dx=float(x - (xmin + xmax) / 2),
                              xml_center_dy=float(y - (ymin + ymax) / 2))
            records.append(record)
    manifest = pd.DataFrame(records)
    if set(manifest.loc[manifest.subset == 'train', 'image_id']) != set(labels['data']):
        raise ValueError('Training images and CSV IDs differ.')
    if manifest.image_id.duplicated().any():
        raise ValueError('Image IDs overlap between train/test.')
    cross = manifest.groupby('pixel_sha256').subset.nunique()
    if (cross > 1).any():
        raise ValueError('Identical pixels occur in both train and test; inspect before continuing.')
    return manifest


def assign_folds(train, seed=42, folds=5):
    """Balance exact-duplicate groups across image resolutions; no test labels used."""
    if folds < 2:
        raise ValueError('Need at least two folds.')
    train = train.copy()
    groups = list(train.groupby('pixel_sha256', sort=True))
    rng = np.random.default_rng(seed)
    buckets = defaultdict(list)
    for digest, group in groups:
        row = group.iloc[0]
        buckets[(int(row.width), int(row.height))].append((digest, len(group)))
    total = np.zeros(folds, dtype=int)
    mapping = {}
    for key in sorted(buckets):
        entries = buckets[key]
        rng.shuffle(entries)
        entries.sort(key=lambda pair: -pair[1])
        sizes = np.zeros(folds, dtype=int)
        for digest, count in entries:
            tie = rng.random(folds)
            fold = min(range(folds), key=lambda f: (sizes[f], total[f], tie[f]))
            mapping[digest] = fold
            sizes[fold] += count
            total[fold] += count
    train['fold'] = train.pixel_sha256.map(mapping)
    if train.groupby('pixel_sha256').fold.nunique().max() != 1:
        raise AssertionError('Duplicate group split across folds.')
    return train


def contact_sheet(data, records, path, predictions=None):
    rows = records.reset_index(drop=True)
    n = min(len(rows), 20)
    canvas = Image.new('RGB', (1120, math.ceil(n / 4) * 232), '#f3f5f7')
    draw = ImageDraw.Draw(canvas)
    font_path = Path('C:/Windows/Fonts/arial.ttf')
    font = ImageFont.truetype(str(font_path), 16) if font_path.exists() else ImageFont.load_default()
    for i, row in rows.iloc[:n].iterrows():
        with Image.open(data / row['subset'] / row.filename) as im:
            im = im.convert('RGB')
            im.thumbnail((276, 195))
            ox, oy = (i % 4) * 280 + (280 - im.width) // 2, (i // 4) * 232
            canvas.paste(im, (ox, oy))
            points = []
            if 'nx' in row and pd.notna(row.nx):
                points.append((row.nx, row.ny, '#00ffff'))
            if predictions is not None:
                points.append((*predictions[i], '#ffff00'))
            for x, y, color in points:
                cx, cy = ox + x * im.width, oy + y * im.height
                draw.ellipse((cx - 4, cy - 4, cx + 4, cy + 4), outline=color, width=2)
                draw.line((cx - 8, cy, cx + 8, cy), fill=color)
                draw.line((cx, cy - 8, cx, cy + 8), fill=color)
        draw.text(((i % 4) * 280 + 7, (i // 4) * 232 + 199),
                  f'{int(row.image_id):04} | {int(row.width)} x {int(row.height)}', font=font, fill='#15202b')
    canvas.save(path)


class FoveaDataset(Dataset):
    def __init__(self, data, records, image_size, augment=False):
        self.records = records.reset_index(drop=True)
        self.image_size = image_size
        self.augment = augment
        self.images = []
        self.jitter = transforms.ColorJitter(brightness=.2, contrast=.2, saturation=.12, hue=.02)
        self.tensor = transforms.Compose([transforms.ToTensor(),
                                          transforms.Normalize([.485, .456, .406], [.229, .224, .225])])
        for row in self.records.itertuples():
            with Image.open(data / row.subset / row.filename) as im:
                self.images.append(im.convert('RGB').resize((image_size, image_size), Image.Resampling.BILINEAR))

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records.iloc[index]
        im = self.images[index].copy()
        point = (np.array([row.nx, row.ny], dtype=np.float32) if row['subset'] == 'train'
                 else np.array([.5, .5], dtype=np.float32))
        if self.augment:
            if random.random() < .5:
                im = im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
                point[0] = 1 - point[0]
            angle, scale = math.radians(random.uniform(-10, 10)), random.uniform(.92, 1.08)
            a, b = scale * math.cos(angle), -scale * math.sin(angle)
            size, center = self.image_size, self.image_size / 2
            tx, ty = random.uniform(-.035, .035) * size, random.uniform(-.035, .035) * size
            matrix = np.array([[a, b, center + tx - a * center - b * center],
                               [-b, a, center + ty + b * center - a * center], [0, 0, 1]])
            point = ((matrix @ np.array([point[0] * size, point[1] * size, 1]))[:2] / size).astype(np.float32)
            if not (point >= 0).all() or not (point <= 1).all():
                raise ValueError('Augmentation moved label outside image.')
            inverse = np.linalg.inv(matrix)[:2].reshape(-1)
            im = im.transform((size, size), Image.Transform.AFFINE, inverse,
                              resample=Image.Resampling.BILINEAR, fillcolor=(0, 0, 0))
            im = self.jitter(im)
        return self.tensor(im), torch.from_numpy(point), torch.tensor([row.width, row.height], dtype=torch.float32)


class DecoderBlock(nn.Module):
    def __init__(self, incoming, skip, outgoing):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(incoming + skip, outgoing, 3, padding=1, bias=False),
                                    nn.GroupNorm(8, outgoing), nn.ReLU(inplace=True),
                                    nn.Conv2d(outgoing, outgoing, 3, padding=1, bias=False),
                                    nn.GroupNorm(8, outgoing), nn.ReLU(inplace=True))

    def forward(self, value, skip):
        value = F.interpolate(value, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.layers(torch.cat([value, skip], dim=1))


def heatmap_loss(logits, points, sigma=1.5):
    height, width = logits.shape[-2:]
    yy = torch.arange(height, device=logits.device, dtype=torch.float32) + .5
    xx = torch.arange(width, device=logits.device, dtype=torch.float32) + .5
    distance = (xx[None, None, :] - points[:, 0, None, None] * width) ** 2
    distance = distance + (yy[None, :, None] - points[:, 1, None, None] * height) ** 2
    target = torch.exp(-distance / (2 * sigma ** 2))
    target = target / target.sum(dim=(-2, -1), keepdim=True).clamp_min(1e-12)
    return -(target.flatten(1) * F.log_softmax(logits.float().flatten(1), dim=1)).sum(1).mean()


def decode_heatmap(logits, radius=3):
    """Local expectation around the peak; cell centers match resize align_corners=False."""
    batch, height, width = logits.shape
    peak = logits.flatten(1).argmax(1)
    py, px = peak // width, peak % width
    yy = torch.arange(height, device=logits.device)[None, :, None]
    xx = torch.arange(width, device=logits.device)[None, None, :]
    mask = (yy - py[:, None, None]).abs() <= radius
    mask = mask & ((xx - px[:, None, None]).abs() <= radius)
    weights = logits.float().masked_fill(~mask, -torch.inf).flatten(1).softmax(1).view(batch, height, width)
    x = (weights * ((xx + .5) / width)).sum((-2, -1))
    y = (weights * ((yy + .5) / height)).sum((-2, -1))
    return torch.stack([x, y], dim=1)


@contextmanager
def training_lock(out):
    handle = (out / 'training.lock').open('a+b')
    if handle.tell() == 0:
        handle.write(b'0')
        handle.flush()
    handle.seek(0)
    try:
        if os.name == 'nt':
            import msvcrt
            msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
        else:
            import fcntl
            fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        yield
    finally:
        handle.close()


In [ ]:
from __future__ import annotations
NOTEBOOK_SOURCE_SHA256 = {'task3_pipeline.py': '5a27db4b7901d185eec831080a16d783ad001adc22b639474d345fe7b315ec3f', 'task3_kaggle_cv.py': '2227703a35b3955b8ce02801930b0bf496703decece4ad5234a1d1f70a0b12e6'}
"""Kaggle-only ResNet34 five-fold comparison: 320, 320+flip TTA, and 512.

Local preparation and tests do not run this module or import PyTorch.
"""

import argparse
import csv
import gc
import hashlib
import json
import math
import os
from pathlib import Path
import random
import shutil
import statistics
import sys
import time
import zipfile


import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models


CV_VERSION = 'resnet34-cv-v1'
WEIGHT_NAME = 'resnet34-b627a593.pth'
WEIGHT_URL = 'https://download.pytorch.org/models/' + WEIGHT_NAME
ARMS = ('resnet34_320', 'resnet34_320_flip', 'resnet34_512')


def stable_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     separators=(',', ':'), allow_nan=False).encode('utf-8')).hexdigest()


def atomic_csv(path, rows, fields=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = list(rows)
    fields = list(fields or (list(rows[0]) if rows else []))
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def read_rows(path):
    with Path(path).open(encoding='utf-8', newline='') as file:
        return list(csv.DictReader(file))


def prediction_rows(records, points, fold, labeled):
    if len(records) != len(points):
        raise ValueError('Prediction count differs from image count.')
    rows, seen = [], set()
    for item, point in zip(records, points):
        key, width, height = int(item['image_id']), int(item['width']), int(item['height'])
        if key in seen:
            raise ValueError('Duplicate image ID.')
        seen.add(key)
        nx, ny = float(point[0]), float(point[1])
        if not all(math.isfinite(v) for v in [nx, ny]) or not (0 <= nx < 1 and 0 <= ny < 1):
            raise ValueError(f'Invalid normalized prediction for image {key}.')
        row = dict(image_id=key, fold=int(fold), width=width, height=height,
                   pixel_sha256=str(item['pixel_sha256']), nx=nx, ny=ny,
                   Fovea_X=nx * width, Fovea_Y=ny * height)
        if labeled:
            x, y = float(item['x']), float(item['y'])
            dx, dy = row['Fovea_X'] - x, row['Fovea_Y'] - y
            row.update(gt_x=x, gt_y=y, dx=dx, dy=dy,
                       distance_px=math.hypot(dx, dy), squared_error=dx * dx + dy * dy)
        rows.append(row)
    return rows


def quantile_linear(values, probability):
    values = sorted(float(v) for v in values)
    if not values:
        raise ValueError('No values.')
    position = (len(values) - 1) * probability
    low, high = math.floor(position), math.ceil(position)
    return values[low] + (values[high] - values[low]) * (position - low)


def metrics_from_rows(rows):
    rows = list(rows)
    if not rows:
        raise ValueError('No labeled predictions.')
    distance, dx, dy = [], [], []
    for row in rows:
        ex = float(row['Fovea_X']) - float(row['gt_x'])
        ey = float(row['Fovea_Y']) - float(row['gt_y'])
        if not math.isfinite(ex) or not math.isfinite(ey):
            raise ValueError('Non-finite error.')
        dx.append(ex)
        dy.append(ey)
        distance.append(math.hypot(ex, ey))
    return dict(n=len(rows), mean_distance_px=statistics.mean(distance),
                median_distance_px=statistics.median(distance),
                p90_distance_px=quantile_linear(distance, .9), max_distance_px=max(distance),
                mae_x_px=statistics.mean(abs(v) for v in dx), mae_y_px=statistics.mean(abs(v) for v in dy),
                rmse_coordinate_px=math.sqrt(sum(x*x+y*y for x, y in zip(dx, dy)) / (2 * len(rows))))


def validate_oof(rows, records):
    expected = {int(r['image_id']): r for r in records}
    if len(expected) != len(records) or len(rows) != len(records):
        raise ValueError('OOF must contain exactly one row per training image.')
    seen, groups = set(), {}
    for row in rows:
        key = int(row['image_id'])
        if key not in expected or key in seen:
            raise ValueError('Unknown or repeated OOF image ID.')
        seen.add(key)
        source = expected[key]
        if int(row['fold']) != int(source['fold']):
            raise ValueError('OOF fold does not match the fixed split.')
        if (int(row['width']), int(row['height'])) != (int(source['width']), int(source['height'])):
            raise ValueError('OOF dimensions differ.')
        if str(row['pixel_sha256']) != str(source['pixel_sha256']):
            raise ValueError('OOF image hash differs.')
        if abs(float(row['gt_x']) - float(source['x'])) > 1e-6 or abs(float(row['gt_y']) - float(source['y'])) > 1e-6:
            raise ValueError('OOF ground truth differs.')
        groups.setdefault(str(source['pixel_sha256']), set()).add(int(row['fold']))
    if seen != set(expected) or any(len(value) != 1 for value in groups.values()):
        raise ValueError('Incomplete OOF coverage or duplicate group leakage.')
    return sorted(rows, key=lambda r: int(r['image_id']))


def validate_test_rows(rows, records, fold):
    expected = {int(r['image_id']): r for r in records}
    if len(rows) != len(expected):
        raise ValueError('Incomplete test prediction coverage.')
    seen = set()
    for row in rows:
        key = int(row['image_id'])
        if key not in expected or key in seen or int(row['fold']) != fold:
            raise ValueError('Invalid test ID/fold.')
        seen.add(key)
        source = expected[key]
        if (int(row['width']), int(row['height'])) != (int(source['width']), int(source['height'])):
            raise ValueError('Test image dimensions differ.')
        if str(row['pixel_sha256']) != str(source['pixel_sha256']):
            raise ValueError('Test image hash differs.')
        x, y = float(row['Fovea_X']), float(row['Fovea_Y'])
        if not all(math.isfinite(v) for v in [x, y]) or not (0 <= x < int(source['width']) and 0 <= y < int(source['height'])):
            raise ValueError('Test prediction outside image.')
    return {int(r['image_id']): r for r in rows}


def average_test_folds(folds, records, fold_count=5):
    if set(folds) != set(range(fold_count)):
        raise ValueError('All five folds are required before exporting a five-fold submission.')
    mappings = [validate_test_rows(folds[f], records, f) for f in range(fold_count)]
    averaged = []
    for source in records:
        key = int(source['image_id'])
        averaged.append(dict(image_id=key, width=int(source['width']), height=int(source['height']),
                             Fovea_X=statistics.mean(float(m[key]['Fovea_X']) for m in mappings),
                             Fovea_Y=statistics.mean(float(m[key]['Fovea_Y']) for m in mappings)))
    return averaged


def write_submission(template_path, predictions, target):
    lookup = {}
    for row in predictions:
        key = int(row['image_id'])
        for axis, dimension in [('X', 'width'), ('Y', 'height')]:
            value = float(row[f'Fovea_{axis}'])
            if not math.isfinite(value) or not 0 <= value < int(row[dimension]):
                raise ValueError('Invalid original-pixel coordinate.')
            image_id = f'{key}_Fovea_{axis}'
            if image_id in lookup:
                raise ValueError('Repeated submission ID.')
            lookup[image_id] = value
    template = read_rows(template_path)
    ids = [row['ImageID'] for row in template]
    if len(ids) != len(set(ids)) or set(ids) != set(lookup):
        raise ValueError('Submission IDs differ from template.')
    rows = [dict(ImageID=key, value=f'{lookup[key]:.6f}') for key in ids]
    atomic_csv(target, rows, ['ImageID', 'value'])
    check = read_rows(target)
    if [r['ImageID'] for r in check] != ids or any(not math.isfinite(float(r['value'])) for r in check):
        raise AssertionError('Submission round trip failed.')


def scaled_heatmap_settings(size):
    if size not in (320, 512):
        raise ValueError('Only the predeclared sizes 320 and 512 are supported.')
    return dict(sigma=1.5 * size / 320, decode_radius=max(1, int(round(3 * size / 320))))


def source_fingerprints():
    if 'NOTEBOOK_SOURCE_SHA256' in globals():
        return dict(NOTEBOOK_SOURCE_SHA256)
    base = Path(__file__).resolve().parent
    return {name: hashlib.sha256((base / name).read_bytes()).hexdigest()
            for name in ['task3_pipeline.py', 'task3_kaggle_cv.py']}


def resolve_weights(output, explicit=None):
    candidates = [Path(explicit)] if explicit else []
    candidates += [Path(torch.hub.get_dir()) / 'checkpoints' / WEIGHT_NAME]
    if '__file__' in globals():
        candidates.append(Path(__file__).resolve().parent.parent / 'task2/.torch/hub/checkpoints' / WEIGHT_NAME)
    if Path('/kaggle/input').exists():
        candidates += sorted(Path('/kaggle/input').rglob(WEIGHT_NAME))
    path = next((p for p in candidates if p.is_file()), None)
    if path is None:
        path = output / '_pretrained' / WEIGHT_NAME
        path.parent.mkdir(parents=True, exist_ok=True)
        if not path.is_file():
            print('Downloading official ResNet34 ImageNet weights (about 87 MB).', flush=True)
            torch.hub.download_url_to_file(WEIGHT_URL, str(path), hash_prefix='b627a593', progress=True)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if not digest.startswith('b627a593'):
        raise ValueError(f'ImageNet weight hash mismatch: {path}')
    save_json(output / 'pretrained_source.json', dict(path=str(path), url=WEIGHT_URL, sha256=digest))
    return path


class ResNet34CV(nn.Module):
    def __init__(self, weights=None):
        super().__init__()
        self.backbone = models.resnet34(weights=None)
        if weights is not None:
            self.backbone.load_state_dict(torch.load(weights, map_location='cpu', weights_only=True), strict=True)
        self.backbone.fc = nn.Identity()
        self.d3 = DecoderBlock(512, 256, 128)
        self.d2 = DecoderBlock(128, 128, 64)
        self.d1 = DecoderBlock(64, 64, 32)
        self.head = nn.Conv2d(32, 1, 1)

    def train(self, mode=True):
        super().train(mode)
        if mode:
            for module in self.backbone.modules():
                if isinstance(module, nn.BatchNorm2d):
                    module.eval()
        return self

    def forward(self, images):
        b = self.backbone
        x = b.maxpool(b.relu(b.bn1(b.conv1(images))))
        c1 = b.layer1(x)
        c2 = b.layer2(c1)
        c3 = b.layer3(c2)
        c4 = b.layer4(c3)
        return self.head(self.d1(self.d2(self.d3(c4, c3), c2), c1))[:, 0]


@torch.inference_mode()
def predict_cv(model, loader, device, radius, flip=False):
    model.eval()
    result = []
    for images, _, _ in loader:
        images = images.to(device)
        with torch.autocast(device_type='cuda', enabled=True):
            coordinates = decode_heatmap(model(images), radius=radius)
            if flip:
                mirrored = decode_heatmap(model(images.flip(-1)), radius=radius)
                mirrored[:, 0] = 1 - mirrored[:, 0]
                coordinates = (coordinates + mirrored) / 2
        result.extend(coordinates.float().cpu().tolist())
    return result


def atomic_torch_save(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(value, temporary)
    temporary.replace(path)


def read_checkpoint(path, signature):
    # Only this experiment's own files/backups are loaded with weights_only=False.
    item = torch.load(path, map_location='cpu', weights_only=False)
    if item.get('signature') != signature:
        raise ValueError(f'Checkpoint protocol mismatch: {path}')
    return item


def current_eta(output, size, fold, epoch, epochs, wall_times, completed_jobs):
    if len(wall_times) < 3:
        return
    rate = statistics.median(wall_times[-8:])
    timing_path = output / 'timing_by_size.json'
    timing = json.loads(timing_path.read_text()) if timing_path.exists() else {}
    timing[str(size)] = dict(seconds_per_epoch=rate, samples=len(wall_times))
    save_json(timing_path, timing)
    estimate = (epochs - epoch) * rate
    estimated_jobs = []
    for candidate_size in [320, 512]:
        known = timing.get(str(candidate_size), {}).get('seconds_per_epoch')
        candidate_rate = known if known is not None else rate * (candidate_size / size) ** 2
        for candidate_fold in range(5):
            key = f'{candidate_size}/{candidate_fold}'
            if key not in completed_jobs and (candidate_size, candidate_fold) != (size, fold):
                estimate += epochs * candidate_rate
                estimated_jobs.append(key)
    information = dict(current_size=size, current_fold=fold, epoch=epoch,
                       median_epoch_seconds=rate, approximate_training_minutes_remaining=estimate / 60,
                       calibration='Measured completed epochs; unmeasured resolution estimated by pixel count.',
                       excludes='Future cold starts, prediction/plotting, downloads and restore overhead.')
    save_json(output / 'eta.json', information)
    if epoch == 3 or epoch % 10 == 0:
        print(f'ETA: about {estimate/60:.1f} training minutes remaining for the full two-size plan.', flush=True)


def train_cv_fold(data, development, valid, folder, config, weights, device, output, completed_jobs):
    folder.mkdir(parents=True, exist_ok=True)
    signature = stable_hash(config)
    config_path = folder / 'train_config.json'
    if config_path.exists() and json.loads(config_path.read_text()) != config:
        raise ValueError(f'Configuration differs: {folder}')
    save_json(config_path, config)
    selected = config['selection_metric']
    complete_path = folder / 'training_complete.json'
    if complete_path.exists():
        done = json.loads(complete_path.read_text())
        if done['signature'] != signature:
            raise ValueError('Completed fold protocol differs.')
        checkpoint = read_checkpoint(folder / f'best_{selected}.pt', signature)
        model = ResNet34CV().to(device)
        model.load_state_dict(checkpoint['model'], strict=True)
        return model, done
    seed_everything(config['seed'])
    torch.set_num_threads(config['cpu_threads'])
    train_set = FoveaDataset(data, development, config['image_size'], augment=True)
    valid_set = FoveaDataset(data, valid, config['image_size'])
    loader = DataLoader(train_set, batch_size=config['batch_size'], shuffle=True, num_workers=0)
    valid_loader = DataLoader(valid_set, batch_size=config['batch_size'], shuffle=False, num_workers=0)
    last_path = folder / 'last.pt'
    model = ResNet34CV(weights=None if last_path.exists() else weights).to(device)
    backbone_ids = {id(p) for p in model.backbone.parameters()}
    optimizer = torch.optim.AdamW([
        dict(params=model.backbone.parameters(), lr=config['lr_backbone']),
        dict(params=[p for p in model.parameters() if id(p) not in backbone_ids], lr=config['lr_decoder'])],
        weight_decay=config['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'],
                                                          eta_min=config['lr_backbone'] * .05)
    scaler = torch.amp.GradScaler('cuda')
    torch.cuda.reset_peak_memory_stats(device=0)
    history, start, wall_times = [], 0, []
    if last_path.exists():
        state = read_checkpoint(last_path, signature)
        model.load_state_dict(state['model'], strict=True)
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        history, start = state['history'], int(state['epoch'])
        random.setstate(state['python_rng'])
        np.random.set_state(state['numpy_rng'])
        torch.set_rng_state(state['torch_rng'])
        torch.cuda.set_rng_state(state['cuda_rng'], device=0)
        del state
        print(f'Resume {config["image_size"]}, fold {config["fold"]}, epoch {start}.', flush=True)
    tracked = ('rmse_coordinate_px', 'mean_distance_px')
    best = {key: dict(score=float('inf'), epoch=0) for key in tracked}
    # Best files can be newer than the optimizer checkpoint after an interruption.
    # Retain these already evaluated, signature-matched candidates rather than mixing scores and weights.
    for key in tracked:
        path = folder / f'best_{key}.pt'
        if path.exists():
            item = read_checkpoint(path, signature)
            best[key] = dict(score=float(item['metrics'][key]), epoch=int(item['epoch']))
            del item
    for epoch in range(start + 1, config['epochs'] + 1):
        begin = time.perf_counter()
        model.train()
        total, count, skipped_steps = 0., 0, 0
        for images, points, _ in loader:
            images, points = images.to(device), points.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', enabled=True):
                logits = model(images)
                if logits.shape != (len(images), config['image_size']//4, config['image_size']//4):
                    raise RuntimeError('Unexpected heatmap dimensions.')
                loss = heatmap_loss(logits, points, sigma=config['sigma'])
            if not torch.isfinite(loss):
                raise RuntimeError('Non-finite heatmap loss.')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.)
            previous_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            skipped_steps += int(scaler.get_scale() < previous_scale)
            total += float(loss.detach()) * len(images)
            count += len(images)
        if skipped_steps == len(loader) and history and history[-1].get('amp_skipped_steps') == len(loader):
            raise RuntimeError('AMP skipped every optimizer step for two consecutive epochs; stop for diagnosis.')
        validation = prediction_rows(valid.to_dict('records'),
                                     predict_cv(model, valid_loader, device, config['decode_radius']),
                                     config['fold'], True)
        metrics = metrics_from_rows(validation)
        for key in tracked:
            if metrics[key] < best[key]['score']:
                best[key] = dict(score=metrics[key], epoch=epoch)
                atomic_torch_save(folder / f'best_{key}.pt', dict(model=model.state_dict(), epoch=epoch,
                    signature=signature, train_config=config, metrics=metrics))
        applied_lr = optimizer.param_groups[0]['lr']
        scheduler.step()
        row = dict(epoch=epoch, train_heatmap_ce=total/count,
                   lr_backbone=applied_lr, **metrics,
                   amp_skipped_steps=skipped_steps,
                   gpu_peak_mib=torch.cuda.max_memory_allocated(device=0)/1024**2,
                   compute_seconds=time.perf_counter()-begin)
        history.append(row)
        if epoch % config['checkpoint_every'] == 0 or epoch == config['epochs']:
            atomic_torch_save(last_path, dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
                scheduler=scheduler.state_dict(), scaler=scaler.state_dict(), epoch=epoch, history=history,
                signature=signature, python_rng=random.getstate(), numpy_rng=np.random.get_state(),
                torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state(device=0)))
        wall = time.perf_counter() - begin
        wall_times.append(wall)
        atomic_csv(folder / 'history.csv', history)
        save_json(folder / 'status.json', dict(state='training', epoch=epoch, epochs=config['epochs'],
                                               best=best, updated_unix=time.time()))
        current_eta(output, config['image_size'], config['fold'], epoch, config['epochs'], wall_times, completed_jobs)
        print(f'{config["image_size"]} fold{config["fold"]} {epoch:03}/{config["epochs"]}'
              f' CE={total/count:.4f} RMSE={metrics["rmse_coordinate_px"]:.2f}px'
              f' MRE={metrics["mean_distance_px"]:.2f}px best={best[selected]["score"]:.2f}px'
              f' AMPskip={skipped_steps} GPUpeak={row["gpu_peak_mib"]:.0f}MiB'
              f' {wall:.1f}s', flush=True)
    done = dict(signature=signature, best=best, selection_metric=selected, epochs=config['epochs'])
    save_json(complete_path, done)
    checkpoint = read_checkpoint(folder / f'best_{selected}.pt', signature)
    model.load_state_dict(checkpoint['model'], strict=True)
    return model, done


def fold_config(protocol, size, fold):
    return dict(cv_version=CV_VERSION, image_size=size, fold=fold,
                seed=protocol['seed'] + fold, split_seed=protocol['seed'],
                epochs=protocol['epochs'], batch_size=protocol['batch_size'],
                lr_backbone=1e-4, lr_decoder=1e-3, weight_decay=1e-4,
                cpu_threads=protocol['cpu_threads'], checkpoint_every=protocol['checkpoint_every'],
                selection_metric=protocol['selection_metric'], data_fingerprint=protocol['data_fingerprint'],
                source_sha256=protocol['source_sha256'], **scaled_heatmap_settings(size))


def run_one_fold(data, train, test, size, fold, output, protocol, weights, completed_jobs, keep_last):
    folder = output / f'resnet34_{size}' / f'fold_{fold}'
    config = fold_config(protocol, size, fold)
    signature = stable_hash(config)
    if (folder / 'fold_complete.json').exists():
        done = json.loads((folder / 'fold_complete.json').read_text())
        if done['signature'] != signature:
            raise ValueError('Cached fold signature differs.')
        for metric in ['rmse_coordinate_px', 'mean_distance_px']:
            if not (folder / f'best_{metric}.pt').is_file():
                raise ValueError('Completed fold is missing a best-model checkpoint.')
        valid = train[train.fold == fold]
        validate_oof(read_rows(folder / 'validation_raw.csv'), valid.to_dict('records'))
        validate_test_rows(read_rows(folder / 'test_raw.csv'), test.to_dict('records'), fold)
        if size == 320:
            validate_oof(read_rows(folder / 'validation_flip.csv'), valid.to_dict('records'))
            validate_test_rows(read_rows(folder / 'test_flip.csv'), test.to_dict('records'), fold)
        print(f'Skip completed {size}, fold{fold}.', flush=True)
        return
    save_json(output / 'status.json', dict(state='running', image_size=size, fold=fold, updated_unix=time.time()))
    model, training = train_cv_fold(data, train[train.fold != fold], train[train.fold == fold],
                                    folder, config, weights, torch.device('cuda:0'), output, completed_jobs)
    valid = train[train.fold == fold]
    valid_loader = DataLoader(FoveaDataset(data, valid, size), batch_size=protocol['batch_size'], shuffle=False, num_workers=0)
    test_loader = DataLoader(FoveaDataset(data, test, size), batch_size=protocol['batch_size'], shuffle=False, num_workers=0)
    metrics = []
    for mode in (['raw', 'flip'] if size == 320 else ['raw']):
        val_points = predict_cv(model, valid_loader, torch.device('cuda:0'), config['decode_radius'], mode == 'flip')
        val_rows = prediction_rows(valid.to_dict('records'), val_points, fold, True)
        atomic_csv(folder / f'validation_{mode}.csv', val_rows)
        test_points = predict_cv(model, test_loader, torch.device('cuda:0'), config['decode_radius'], mode == 'flip')
        atomic_csv(folder / f'test_{mode}.csv', prediction_rows(test.to_dict('records'), test_points, fold, False))
        metrics.append(dict(arm=f'resnet34_{size}' + ('_flip' if mode == 'flip' else ''),
                            fold=fold, best_epoch=training['best'][protocol['selection_metric']]['epoch'],
                            **metrics_from_rows(val_rows)))
    atomic_csv(folder / 'fold_metrics.csv', metrics)
    save_json(folder / 'fold_complete.json', dict(signature=signature, training=training,
                                                 predictions_complete=True, updated_unix=time.time()))
    # Remove only this completed run's own optimizer checkpoint, keeping both best model files.
    if not keep_last and (folder / 'last.pt').exists():
        (folder / 'last.pt').unlink()
    del model, valid_loader, test_loader
    gc.collect()
    torch.cuda.empty_cache()


def build_reports(data, train, test, output, protocol):
    comparisons, fold_metrics, available, epochs = [], [], [], {}
    for arm in ARMS:
        size = 512 if arm.endswith('512') else 320
        mode = 'flip' if arm.endswith('flip') else 'raw'
        folders = [output / f'resnet34_{size}' / f'fold_{fold}' for fold in range(5)]
        if not all((folder / 'fold_complete.json').exists() for folder in folders):
            continue
        oof, test_folds = [], {}
        epochs[arm] = []
        for fold, folder in enumerate(folders):
            done = json.loads((folder / 'fold_complete.json').read_text())
            if done['signature'] != stable_hash(fold_config(protocol, size, fold)):
                raise ValueError('Report found mismatched completed fold.')
            oof.extend(read_rows(folder / f'validation_{mode}.csv'))
            test_folds[fold] = read_rows(folder / f'test_{mode}.csv')
            for row in read_rows(folder / 'fold_metrics.csv'):
                if row['arm'] == arm:
                    fold_metrics.append(row)
                    epochs[arm].append(int(row['best_epoch']))
        oof = validate_oof(oof, train.to_dict('records'))
        atomic_csv(output / f'oof_{arm}.csv', oof)
        averaged = average_test_folds(test_folds, test.to_dict('records'))
        atomic_csv(output / f'test_{arm}_5fold.csv', averaged)
        write_submission(data / 'sample_submission.csv', averaged, output / f'submission_{arm}_5fold.csv')
        comparisons.append(dict(arm=arm, **metrics_from_rows(oof)))
        available.append(arm)
        worst_ids = [int(r['image_id']) for r in sorted(oof, key=lambda r: float(r['distance_px']), reverse=True)[:20]]
        ordered = train.set_index('image_id').loc[worst_ids].reset_index()
        lookup = {int(r['image_id']): r for r in oof}
        points = [[float(lookup[key]['Fovea_X']) / float(lookup[key]['width']),
                   float(lookup[key]['Fovea_Y']) / float(lookup[key]['height'])] for key in worst_ids]
        contact_sheet(data, ordered, output / f'worst_examples_{arm}.png', points)
    if comparisons:
        comparisons.sort(key=lambda r: r[protocol['selection_metric']])
        atomic_csv(output / 'comparison.csv', comparisons)
        atomic_csv(output / 'fold_metrics.csv', fold_metrics)
        print('\nOOF comparison (development scores):', flush=True)
        print(pd.DataFrame(comparisons).to_string(index=False), flush=True)
    complete = set(available) == set(ARMS)
    selection = dict(complete=complete, available_arms=available, metric=protocol['selection_metric'],
                      official_metric_confirmed=False, fold_epochs=epochs,
                      limitation='Fold stopping and arm choice use validation labels; OOF is development evaluation, not an independent test score.',
                      submission_models='Equal-weight ensemble of five 64-image-training fold checkpoints; no full-data refit.',
                      submitted_to_competition=False)
    if complete:
        selection['selected_arm'] = comparisons[0]['arm']
        source = output / f'submission_{comparisons[0]["arm"]}_5fold.csv'
        shutil.copy2(source, output / 'submission_selected_5fold.csv')
    save_json(output / 'selection.json', selection)
    return complete


def restore_backup(source, output, protocol):
    if source is None:
        return
    source = Path(source).resolve()
    if source == output.resolve():
        return
    candidate = source / 'protocol.json'
    if not candidate.is_file() or json.loads(candidate.read_text()) != protocol:
        raise ValueError('Backup protocol/data/code differs. Choose the correct prior output folder.')
    if (output / 'protocol.json').exists():
        print('Current output already has progress; resuming it without merging backup files.', flush=True)
        return
    for path in source.rglob('*'):
        if not path.is_file() or path.is_symlink() or path.suffix in {'.tmp', '.zip'}:
            continue
        if not path.resolve().is_relative_to(source):
            raise ValueError('Backup path escapes its source directory.')
        destination = output / path.relative_to(source)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, destination)


def light_archive(output):
    target = output / 'task3_cv_results_light.zip'
    temporary = output / 'task3_cv_results_light.zip.tmp'
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in output.rglob('*'):
            if path.is_file() and path.suffix in {'.json', '.csv', '.png', '.md'}:
                archive.write(path, path.relative_to(output))
    temporary.replace(target)


def run_cv(data_dir=None, output_dir='/kaggle/working/task3_resnet34_cv', resume_dir=None,
           weights=None, epochs=80, batch_size=8, seed=42, selection_metric='rmse_coordinate_px',
           checkpoint_every=5, cpu_threads=2, keep_last=False, run_sizes=(320, 512), run_folds=(0, 1, 2, 3, 4),
           allow_local=False):
    if not Path('/kaggle/working').exists() and not allow_local:
        raise RuntimeError('Use this Notebook on Kaggle; local training is disabled by default.')
    if not torch.cuda.is_available():
        raise RuntimeError('Kaggle GPU is not enabled; stop before CPU training.')
    if selection_metric not in ('rmse_coordinate_px', 'mean_distance_px'):
        raise ValueError('Unsupported selection metric.')
    if epochs < 1 or batch_size < 1 or checkpoint_every < 1:
        raise ValueError('Epoch, batch, checkpoint interval must be positive.')
    if not run_sizes or not set(run_sizes) <= {320, 512} or len(set(run_sizes)) != len(run_sizes):
        raise ValueError('Invalid resolution schedule.')
    if not run_folds or not set(run_folds) <= set(range(5)) or len(set(run_folds)) != len(run_folds):
        raise ValueError('Invalid fold schedule.')
    data, output = find_data(data_dir), Path(output_dir).resolve()
    output.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(output).free < 3 * 1024**3:
        raise RuntimeError('Need at least 3 GiB free for weights, temporary checkpoint replacement and results.')
    manifest = load_manifest(data)
    train = assign_folds(manifest[manifest.subset == 'train'], seed, 5)
    test = manifest[manifest.subset == 'test'].copy()
    if len(train) != 80 or len(test) != 20:
        raise ValueError('Unexpected Task3 image counts.')
    protocol = dict(version=CV_VERSION, seed=seed, folds=5, epochs=epochs, batch_size=batch_size,
                    cpu_threads=cpu_threads, checkpoint_every=checkpoint_every, selection_metric=selection_metric,
                    data_fingerprint=hashlib.sha256(train.to_csv(index=False).encode()).hexdigest(),
                    test_fingerprint=hashlib.sha256(test.to_csv(index=False).encode()).hexdigest(),
                    source_sha256=source_fingerprints(), plan_sizes=[320, 512], plan_arms=list(ARMS),
                    gaussian_policy='sigma=1.5*(size/320); radius=round(3*size/320)',
                    tta_policy='Horizontal flip, inverse x, equal coordinate average; same raw-selected checkpoint.',
                    augmentation='Same FoveaDataset geometric/color augmentations and ImageNet normalization as prior baseline.')
    restore_backup(resume_dir, output, protocol)
    protocol_path = output / 'protocol.json'
    if protocol_path.exists() and json.loads(protocol_path.read_text()) != protocol:
        raise ValueError('Existing run configuration differs. Use a new output directory.')
    save_json(protocol_path, protocol)
    train.to_csv(output / 'split.csv', index=False)
    manifest.to_csv(output / 'manifest.csv', index=False)
    save_json(output / 'environment.json', dict(torch=torch.__version__, numpy=np.__version__,
        gpu=torch.cuda.get_device_name(0), visible_gpu_count=torch.cuda.device_count(),
        device_policy='cuda:0, folds sequential, no DataParallel', cpu_threads=cpu_threads, workers=0))
    print(f'{len(train)} labeled images, {train.pixel_sha256.nunique()} pixel groups, 5 folds.', flush=True)
    print('GPU:', torch.cuda.get_device_name(0), '| using cuda:0 only | outputs:', output, flush=True)
    weights = resolve_weights(output, weights)
    completed_jobs = {f'{size}/{fold}' for size in (320, 512) for fold in range(5)
                      if (output / f'resnet34_{size}/fold_{fold}/fold_complete.json').exists()}
    with training_lock(output):
        try:
            for size in run_sizes:
                for fold in run_folds:
                    run_one_fold(data, train, test, size, fold, output, protocol, weights, completed_jobs, keep_last)
                    completed_jobs.add(f'{size}/{fold}')
                build_reports(data, train, test, output, protocol)
                light_archive(output)
            complete = build_reports(data, train, test, output, protocol)
            save_json(output / 'status.json', dict(state='completed' if complete else 'partial_completed',
                                                   completed_jobs=sorted(completed_jobs), updated_unix=time.time()))
            light_archive(output)
        except BaseException as error:
            save_json(output / 'status.json', dict(state='interrupted' if isinstance(error, KeyboardInterrupt) else 'failed',
                                                   error=str(error), updated_unix=time.time()))
            light_archive(output)
            raise
    return output


## 3. 运行十个模型并生成折外比较

前几轮会记录实际耗时并估算完整计划的剩余训练时间；未测过的分辨率暂按像素数估计，因此 ETA 是近似值，不包含下载、冷启动和后处理。各折逐轮输出 RMSE/MRE、AMP 跳过步数、峰值显存，训练历史保留在对应目录。

每五轮保存可续训的模型、优化器、调度器与随机数状态；每次改善均保存相应最佳模型。完整一折结束后，默认删除该折 `last.pt` 以节省空间，保留 `best_rmse_coordinate_px.pt` 和 `best_mean_distance_px.pt`，并直接复用已生成的预测。


In [ ]:
RESULT_DIR = run_cv(
    data_dir=DATA_DIR, output_dir=OUTPUT_DIR, resume_dir=RESUME_DIR, weights=WEIGHTS,
    epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED, selection_metric=SELECTION_METRIC,
    checkpoint_every=CHECKPOINT_EVERY, cpu_threads=CPU_THREADS,
    keep_last=KEEP_LAST_CHECKPOINT, run_sizes=RUN_SIZES, run_folds=RUN_FOLDS,
)


## 4. 结果、误差与提交候选

`comparison.csv` 使用完整80张折外预测整体计算指标，**不把五个折的 RMSE 简单平均**。`fold_metrics.csv` 展示折间差异；`oof_*.csv` 保存每张图的原图坐标、误差和所属折。大误差图片会单独生成可视化。

只有一个方案的五折齐全，才导出其五折测试集集成 CSV；三种方案全部齐全后才生成 `submission_selected_5fold.csv`。测试预测直接平均五份折模型，**不进行全量重训**。这五个模型各训练64张图，与此前80张全量版不同。

轮次和方案选择仍参考验证标签，因此 OOF 是开发集评估，不是完全独立的泛化成绩。候选 CSV 均按40行模板导出，本 Notebook 不执行比赛提交。保留原有12.31624版本作为对照，等待真实结果再决定下一步。


In [ ]:
from IPython.display import display, Image as DisplayImage
if (RESULT_DIR / 'comparison.csv').exists():
    display(pd.read_csv(RESULT_DIR / 'comparison.csv'))
    display(pd.read_csv(RESULT_DIR / 'fold_metrics.csv'))
print((RESULT_DIR / 'selection.json').read_text(encoding='utf-8'))
for arm in ARMS:
    image_path = RESULT_DIR / f'worst_examples_{arm}.png'
    if image_path.exists():
        print(arm, '：青色标签、黄色预测')
        display(DisplayImage(filename=str(image_path)))
selected = RESULT_DIR / 'submission_selected_5fold.csv'
if selected.exists():
    display(pd.read_csv(selected))
    print('Selected candidate:', selected)
print('Light results ZIP:', RESULT_DIR / 'task3_cv_results_light.zip')


## 5. 保存和恢复

- 完成后下载 `task3_cv_results_light.zip`，包含表格、配置与图片，**不含模型权重或优化器**。
- 保留当前 Kaggle Notebook 版本的完整 Output。恢复时把此前 Output 作为输入挂载，将 `RESUME_DIR` 指向含 `protocol.json` 的目录；数据、代码、训练参数一致才接受恢复。
- 当前输出目录已有进度时优先续跑当前进度，不混合另一份备份。中断前完整写入的断点可以恢复；未完整写入的 `.tmp` 不作为断点。
- 断点和模型均是本实验自己生成的文件。只挂载你自己的可信实验输出。
